# Fase 2: Extração Bruta para o IVS Multidimensional (Censo 2022)

Este notebook é o primeiro passo do pipeline de ETL da nova metodologia. O seu objetivo é **puramente extrativo e de validação**.
1. Lê as variáveis alvo de 8 arquivos modulares do Censo (incluindo as novas bases de Demografia e Parentesco).
2. Preserva os denominadores globais originais.
3. Cria a classificação de **Morfologia Urbana** (Tipo de Moradia Predominante).
4. Realiza uma auditoria automática para atestar a integridade do Join.

In [ ]:
import pandas as pd  # Importa a biblioteca pandas para manipulação de dados
import numpy as np    # Importa a biblioteca numpy para operações numéricas
import os             # Importa a biblioteca os para operações com o sistema de arquivos

print("1. Configurando ambiente e mapeando variáveis...")  # Mensagem de status

caminho_dados = '../../dados/'  # Define o caminho para a pasta de dados
caminho_bd = '../../banco_de_dados/'  # Define o caminho para a pasta de saída dos bancos de dados
os.makedirs(caminho_bd, exist_ok=True)  # Cria a pasta de saída se não existir

# Mapeamento cirúrgico das colunas aprovadas
col_basico = ['CD_SETOR', 'NM_MUN', 'NM_BAIRRO', 'SITUACAO', 'v0001']  # Colunas do arquivo básico
col_dom1 = ['CD_setor', 'V00001', 'V00002', 'V00005', 'V00006', 'V00047', 'V00048', 'V00049', 'V00050', 'V00051', 'V00052']  # Colunas do domicílio 1
col_dom2 = ['setor', 'V00112', 'V00113', 'V00114', 'V00115', 'V00116', 'V00117', 'V00118', 'V00312', 'V00313', 'V00314', 'V00315', 'V00316', 'V00398', 'V00399', 'V00400', 'V00401', 'V00402', 'V00236', 'V00238']  # Colunas do domicílio 2
col_alfab = ['CD_setor', 'V00900', 'V00901']  # Colunas de alfabetização
col_raca = ['CD_SETOR', 'V01318', 'V01320', 'V01321']  # Colunas de cor ou raça
col_renda = ['CD_SETOR', 'V06004']  # Colunas de renda
col_demog = ['CD_setor', 'V01031', 'V01032', 'V01033']  # Colunas de demografia
col_parent = ['CD_SETOR', 'V01042']  # Colunas de parentesco

print("2. Lendo os arquivos CSV para a memória (Isto pode demorar alguns segundos)...")  # Mensagem de status

def ler_csv_padronizado(caminho_arquivo, sep, dtype, usecols, encoding_list=['utf-8', 'latin1', 'cp1252'], rename_cols=None):
    # Função para ler CSV testando múltiplos encodings e renomeando colunas se necessário
    for enc in encoding_list:
        try:
            df = pd.read_csv(caminho_arquivo, sep=sep, dtype=dtype, usecols=usecols, encoding=enc, low_memory=False)  # Tenta ler o arquivo com o encoding atual
            if rename_cols:
                df = df.rename(columns=rename_cols)  # Renomeia colunas se necessário
            print(f"Arquivo {os.path.basename(caminho_arquivo)} lido com encoding: {enc}")  # Informa o encoding usado
            return df
        except UnicodeDecodeError:
            continue  # Se der erro de encoding, tenta o próximo
    raise UnicodeDecodeError(f"Não foi possível ler {caminho_arquivo} com os encodings testados.")  # Se nenhum encoding funcionar, lança erro

df_basico = ler_csv_padronizado(caminho_dados + 'Agregados_por_setores_basico_BR_20250417.csv', sep=';', dtype=str, usecols=col_basico)  # Lê o arquivo básico
df_dom1 = ler_csv_padronizado(caminho_dados + 'Agregados_por_setores_caracteristicas_domicilio1_BR.csv', sep=';', dtype=str, usecols=col_dom1, rename_cols={'CD_setor': 'CD_SETOR'})  # Lê domicílio 1 e padroniza coluna-chave
df_dom2 = ler_csv_padronizado(caminho_dados + 'Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.csv', sep=';', dtype=str, usecols=col_dom2, rename_cols={'setor': 'CD_SETOR'})  # Lê domicílio 2 e padroniza coluna-chave
df_alfab = ler_csv_padronizado(caminho_dados + 'Agregados_por_setores_alfabetizacao_BR.csv', sep=';', dtype=str, usecols=col_alfab, rename_cols={'CD_setor': 'CD_SETOR'})  # Lê alfabetização e padroniza coluna-chave
df_raca = ler_csv_padronizado(caminho_dados + 'Agregados_por_setores_cor_ou_raca_BR.csv', sep=';', dtype=str, usecols=col_raca)  # Lê cor ou raça
df_renda = ler_csv_padronizado(caminho_dados + 'Agregados_por_setores_renda_responsavel_BR.csv', sep=';', dtype=str, usecols=col_renda)  # Lê renda
df_demog = ler_csv_padronizado(caminho_dados + 'Agregados_por_setores_demografia_BR.csv', sep=';', dtype=str, usecols=col_demog, rename_cols={'CD_setor': 'CD_SETOR'})  # Lê demografia e padroniza coluna-chave
df_parent = ler_csv_padronizado(caminho_dados + 'Agregados_por_setores_parentesco_BR.csv', sep=';', dtype=str, usecols=col_parent)  # Lê parentesco

print("Leitura concluída!")  # Mensagem de status

1. Configurando ambiente e mapeando variáveis...
2. Lendo os arquivos CSV para a memória (Isto pode demorar alguns segundos)...
Arquivo Agregados_por_setores_basico_BR_20250417.csv lido com encoding: latin1
Arquivo Agregados_por_setores_caracteristicas_domicilio1_BR.csv lido com encoding: utf-8
Arquivo Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.csv lido com encoding: utf-8
Arquivo Agregados_por_setores_alfabetizacao_BR.csv lido com encoding: utf-8
Arquivo Agregados_por_setores_cor_ou_raca_BR.csv lido com encoding: utf-8
Arquivo Agregados_por_setores_renda_responsavel_BR.csv lido com encoding: utf-8
Arquivo Agregados_por_setores_demografia_BR.csv lido com encoding: utf-8
Arquivo Agregados_por_setores_parentesco_BR.csv lido com encoding: utf-8
Leitura concluída!


In [18]:
import os
print("Célula 3: Lendo os arquivos CSV originais (apenas as colunas necessárias)...")
print("Isto protegerá a memória RAM do computador.")

# Função para leitura robusta, similar ao notebook anterior

def ler_csv_padronizado(caminho_arquivo, sep, dtype, usecols, encoding='latin1', rename_cols=None):
    df = pd.read_csv(caminho_arquivo, sep=sep, dtype=dtype, usecols=usecols, encoding=encoding, low_memory=False)
    if rename_cols:
        df = df.rename(columns=rename_cols)
    return df

# Caminhos dos arquivos
arq_basico = os.path.join(caminho_dados, 'Agregados_por_setores_basico_BR_20250417.csv')
arq_dom1 = os.path.join(caminho_dados, 'Agregados_por_setores_caracteristicas_domicilio1_BR.csv')
arq_dom2 = os.path.join(caminho_dados, 'Agregados_por_setores_caracteristicas_domicilio2_BR_20250417.csv')
arq_alfab = os.path.join(caminho_dados, 'Agregados_por_setores_alfabetizacao_BR.csv')
arq_raca = os.path.join(caminho_dados, 'Agregados_por_setores_cor_ou_raca_BR.csv')
arq_renda = os.path.join(caminho_dados, 'Agregados_por_setores_renda_responsavel_BR.csv')
arq_demog = os.path.join(caminho_dados, 'Agregados_por_setores_demografia_BR.csv')
arq_parent = os.path.join(caminho_dados, 'Agregados_por_setores_parentesco_BR.csv')

# Leitura dos arquivos, padronizando nomes de colunas-chave

df_basico = ler_csv_padronizado(arq_basico, sep=';', dtype=str, usecols=col_basico)
df_dom1 = ler_csv_padronizado(arq_dom1, sep=';', dtype=str, usecols=col_dom1, rename_cols={'CD_setor': 'CD_SETOR'})
df_dom2 = ler_csv_padronizado(arq_dom2, sep=';', dtype=str, usecols=col_dom2, rename_cols={'setor': 'CD_SETOR'})
df_alfab = ler_csv_padronizado(arq_alfab, sep=';', dtype=str, usecols=col_alfab, rename_cols={'CD_setor': 'CD_SETOR'})
df_raca = ler_csv_padronizado(arq_raca, sep=';', dtype=str, usecols=col_raca)
df_renda = ler_csv_padronizado(arq_renda, sep=';', dtype=str, usecols=col_renda)
df_demog = ler_csv_padronizado(arq_demog, sep=';', dtype=str, usecols=col_demog, rename_cols={'CD_setor': 'CD_SETOR'})
df_parent = ler_csv_padronizado(arq_parent, sep=';', dtype=str, usecols=col_parent)

print("Leitura concluída!")

Célula 3: Lendo os arquivos CSV originais (apenas as colunas necessárias)...
Isto protegerá a memória RAM do computador.
Leitura concluída!


In [ ]:
print("3. Unificando a base de dados (Merge)...")  # Mensagem de status

# Realiza o merge (junção) de todos os DataFrames lidos, usando a coluna CD_SETOR como chave
df_bruto = df_basico.merge(df_dom1, on='CD_SETOR', how='left')  # Junta dom1
df_bruto = df_bruto.merge(df_dom2, on='CD_SETOR', how='left')  # Junta dom2
df_bruto = df_bruto.merge(df_alfab, on='CD_SETOR', how='left')  # Junta alfabetização
df_bruto = df_bruto.merge(df_raca, on='CD_SETOR', how='left')  # Junta cor/raça
df_bruto = df_bruto.merge(df_renda, on='CD_SETOR', how='left')  # Junta renda
df_bruto = df_bruto.merge(df_demog, on='CD_SETOR', how='left')  # Junta demografia
df_bruto = df_bruto.merge(df_parent, on='CD_SETOR', how='left')  # Junta parentesco

print("4. Criando o classificador de Morfologia Urbana...")  # Mensagem de status

colunas_moradia = ['V00047', 'V00048', 'V00049', 'V00050', 'V00051', 'V00052']  # Lista das colunas de tipos de moradia
df_morfologia_temp = df_bruto[colunas_moradia].apply(pd.to_numeric, errors='coerce').fillna(0)  # Converte para número, tratando 'X' como 0

dicionario_morfologia = {  # Dicionário para traduzir código em nome legível
    'V00047': 'Casa',
    'V00048': 'Casa de Vila/Condomínio',
    'V00049': 'Apartamento',
    'V00050': 'Cortiço/Casa de Cômodos',
    'V00051': 'Maloca Indígena',
    'V00052': 'Estrutura Degradada/Inacabada'
}

coluna_maxima = df_morfologia_temp.idxmax(axis=1)  # Encontra a coluna com maior valor em cada linha
df_bruto['Moradia_Predominante'] = coluna_maxima.map(dicionario_morfologia)  # Atribui o tipo predominante

linhas_zeradas = df_morfologia_temp.sum(axis=1) == 0  # Identifica linhas sem moradia registrada
df_bruto.loc[linhas_zeradas, 'Moradia_Predominante'] = 'Indefinido/Sem Moradia'  # Marca como indefinido

print("Base unificada e classificada!")  # Mensagem de status

3. Unificando a base de dados (Merge)...
4. Criando o classificador de Morfologia Urbana...
Base unificada e classificada!


In [ ]:
# Só executa a exportação se você confirmar que a auditoria acima passou com sucesso
print("5. Exportando a Base Bruta Multidimensional Auditada...")  # Mensagem de status

caminho_saida = caminho_bd + 'Base_Bruta_Multidimensional_Censo2022.csv'  # Define o caminho do arquivo de saída
df_bruto.to_csv(caminho_saida, index=False, sep=';', encoding='utf-8-sig')  # Exporta o DataFrame para CSV

print(f"Processo finalizado com excelência! Ficheiro guardado em:\n{caminho_saida}")  # Mensagem de sucesso

5. Exportando a Base Bruta Multidimensional Auditada...
Processo finalizado com excelência! Ficheiro guardado em:
../../banco_de_dados/Base_Bruta_Multidimensional_Censo2022.csv


In [ ]:
print("=========================================")  # Separador visual
print("      AUDITORIA DE INTEGRIDADE ETL       ")  # Título da auditoria
print("=========================================")

erros_encontrados = 0  # Inicializa o contador de erros

# 1. Validação de Linhas (Garante que nenhum setor foi duplicado ou perdido no JOIN)
linhas_esperadas = len(df_basico)  # Número de linhas esperadas
linhas_obtidas = len(df_bruto)      # Número de linhas obtidas após o merge
if linhas_esperadas == linhas_obtidas:
    print(f"[OK] Total de Setores mantido: {linhas_obtidas}")  # OK se igual
else:
    print(f"[ERRO] O número de linhas mudou! Esperado: {linhas_esperadas} | Obtido: {linhas_obtidas}")  # ERRO se diferente
    erros_encontrados += 1

# 2. Validação da Chave Primária (Garante que o código do setor não corrompeu)
nulos_chave = df_bruto['CD_SETOR'].isnull().sum()  # Conta valores nulos na chave
if nulos_chave == 0:
    print("[OK] Chave Primária (CD_SETOR) está íntegra e sem valores nulos.")  # OK se não há nulos
else:
    print(f"[ERRO] Foram encontrados {nulos_chave} setores sem código de identificação!")  # ERRO se há nulos
    erros_encontrados += 1

# 3. Validação da Inclusão dos Novos Arquivos (Prova que leu Demografia e Parentesco)
colunas_novas = ['V01031', 'V01042', 'Moradia_Predominante']  # Colunas essenciais esperadas
colunas_faltantes = [col for col in colunas_novas if col not in df_bruto.columns]  # Verifica se faltam
if not colunas_faltantes:
    print(f"[OK] Novas colunas essenciais foram integradas com sucesso: {colunas_novas}")  # OK se todas presentes
else:
    print(f"[ERRO] Colunas não encontradas após o Join: {colunas_faltantes}")  # ERRO se faltar
    erros_encontrados += 1

# 4. Validação do Sigilo Original (Garante que não limpamos os 'X' acidentalmente)
setores_com_sigilo = df_bruto[df_bruto.eq('X').any(axis=1)].shape[0]  # Conta linhas com 'X'
if setores_com_sigilo > 0:
    print(f"[OK] Os dados originais do IBGE estão preservados ({setores_com_sigilo} setores contêm marcações de sigilo 'X').")  # OK se há sigilo
else:
    print("[ALERTA] Nenhum dado sigiloso 'X' foi encontrado. Verifique se as variáveis não foram convertidas para número antes da hora.")  # Alerta se não há sigilo

print("=========================================")  # Separador visual
if erros_encontrados == 0:
    print("🟢 STATUS: BASE APROVADA PARA EXPORTAÇÃO.")  # OK final
else:
    print(f"🔴 STATUS: {erros_encontrados} ERROS CRÍTICOS ENCONTRADOS. PARE O PROCESSO.")  # ERRO final

      AUDITORIA DE INTEGRIDADE ETL       
[OK] Total de Setores mantido: 468099
[OK] Chave Primária (CD_SETOR) está íntegra e sem valores nulos.
[OK] Novas colunas essenciais foram integradas com sucesso: ['V01031', 'V01042', 'Moradia_Predominante']
[OK] Os dados originais do IBGE estão preservados (398239 setores contêm marcações de sigilo 'X').
🟢 STATUS: BASE APROVADA PARA EXPORTAÇÃO.


: 